In [1]:
# ===============================
# 1. Importar librerías
# ===============================

from pathlib import Path
from time import sleep
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
ruta = r"C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\silver\df_tilos_limpio.parquet"
df = pd.read_parquet(ruta)

In [3]:
# ===============================
# 1. Detectar portal
# ===============================

def detectar_portal(detail_url):
    """
    Identifica el portal de contratación a partir de la URL.
    """
    if pd.isna(detail_url):
        return "sin_url"

    detail_url = str(detail_url).lower()

    if "contratosdegalicia.gal" in detail_url:
        return "galicia"

    if "contratos-publicos.comunidad.madrid" in detail_url:
        return "madrid"

    if "contrataciondelestado.es" in detail_url:
        return "contratacion_estado"

    return "otro"


df["portal"] = df["detail_url"].apply(detectar_portal)

df["portal"].value_counts(dropna=False)

portal
galicia                11787
otro                     705
madrid                    89
contratacion_estado        8
Name: count, dtype: int64

In [5]:
# ===============================
# 3. Muestra estratificada de 10 licitaciones
# ===============================

distribucion_muestra = {
    "galicia": 3,
    "madrid": 3,
    "contratacion_estado": 4
}

muestras = []

for portal, n in distribucion_muestra.items():

    df_portal = df[df["portal"] == portal].copy()

    if df_portal.empty:
        print(f"No hay registros para el portal: {portal}")
        continue

    n_disponible = min(n, len(df_portal))

    muestra_portal = df_portal.sample(
        n=n_disponible,
        random_state=42
    )

    muestras.append(muestra_portal)

muestra = pd.concat(
    muestras,
    ignore_index=True
)

muestra.shape

(10, 29)

In [6]:
# ===============================
# 4. Validar distribución final
# ===============================

muestra["portal"].value_counts(dropna=False)

portal
contratacion_estado    4
galicia                3
madrid                 3
Name: count, dtype: int64

In [7]:
# ===============================
# 5. Revisar muestra
# ===============================

muestra[
    [
        "licitacion_id",
        "titulo",
        "detail_url",
        "portal"
    ]
]

,licitacion_id,titulo,detail_url,portal
0,4f6977378753d434,"Adquisición de diversos artículos, actuaciones...",https://www.contratosdegalicia.gal/licitacion?...,galicia
1,24dacf3b8e9cf898,"Adquisición de diversos artículos, actuaciones...",https://www.contratosdegalicia.gal/licitacion?...,galicia
2,355922868d871272,003730 - mrsa screen agar - placa,https://www.contratosdegalicia.gal/licitacion?...,galicia
3,6e2fd20856764985,P.A.S.A. 43/2024. Servicio para la realización...,https://contratos-publicos.comunidad.madrid/co...,madrid
4,da0895a330e4a7a8,Servicio realización RM para Laboratorio de Im...,https://contratos-publicos.comunidad.madrid/co...,madrid
5,974ece9fc0483c8e,Servicio de realización de pruebas de laborato...,https://contratos-publicos.comunidad.madrid/co...,madrid
6,ddec9f7e2e9944dc,Servicio de reconocimientos médicos personal a...,https://contrataciondelestado.es/FileSystem/se...,contratacion_estado
7,41db3a58bd5c3bbd,Servicio Especializado de Rehabilitación Hospi...,https://contrataciondelestado.es/wps/poc?uri=d...,contratacion_estado
8,1967eade2fccb129,Contratación del servicio de Asistencia Sanita...,https://contrataciondelestado.es/wps/portal/%2...,contratacion_estado
9,bcf984626102e175,Servicio de prevención ajeno en las especialid...,https://contrataciondelestado.es/wps/poc?uri=d...,contratacion_estado


In [9]:
# ===============================
# Guardar muestra estratificada en Silver
# ===============================

from pathlib import Path

ruta_silver = Path(
    r"C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\silver"
)

ruta_silver.mkdir(parents=True, exist_ok=True)

muestra.to_parquet(
    ruta_silver / "muestra_10_estratificada_3_portales.parquet",
    index=False
)

In [40]:
# ===============================
# Validar archivo guardado
# ===============================

muestra_validada = pd.read_parquet(
    ruta_silver / "muestra_10_estratificada_3_portales.parquet"
)

muestra_validada["portal"].value_counts()

portal
contratacion_estado    4
galicia                3
madrid                 3
Name: count, dtype: int64

In [41]:
muestra = muestra_validada.copy()

In [42]:
muestra.columns

Index(['licitacion_id', 'titulo', 'detail_url', 'updated', 'expediente',
       'tipo_contrato_codigo', 'lugar_ejecucion_codigo', 'cpv_codes',
       'organo_contratacion', 'estado_codigo', 'fecha_publicacion',
       'procedimiento_codigo', 'importe_sin_impuestos', 'fuente_publicacion',
       'presentacion_hasta', 'presentacion_hora', 'notice_types',
       'organo_dir3', 'contrato_duracion', 'contrato_duracion_unidad',
       'ofertas_recibidas', 'adjudicatario', 'adjudicatario_nif', 'estado',
       'tipo_contrato', 'url', 'cpv_descripcion', 'cpv_nivel', 'portal'],
      dtype='object')

In [43]:
muestra.iloc[7]["detail_url"]

'https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=rsl%2BfImh9a6LAncw3qdZkA%3D%3D'

In [44]:
# ===============================
# 9. Seleccionar contratacion_estado
# ===============================

muestra_contratacion_estado = muestra[muestra["portal"] == "contratacion_estado"].copy()

licitacion_contratacion_estado= muestra_contratacion_estado.iloc[0]

licitacion_id_contratacion_estado = licitacion_contratacion_estado["licitacion_id"]
url_contratacion_estado = licitacion_contratacion_estado["detail_url"]

print("Licitación:", licitacion_id_contratacion_estado)
print("URL:", url_contratacion_estado)

Licitación: ddec9f7e2e9944dc
URL: https://contrataciondelestado.es/FileSystem/servlet/GetDocumentByIdServlet?DocumentIdParam=DYMMp2mFqXz4PkjvZh9I5vyWLItwtIt0n8yxEgEwqVaUFQkv5PXRMxoYGegc0NMYm0UHT%2FoXEFKTkr37fnFcEQrX2pafwfyc3CjQTJA0783VTD3T98KC0nSgQFM0q%2B5s&cifrado=QUC1GjXXSiLkydRHJBmbpw%3D%3D


In [45]:
# ===============================
# 10. Diagnóstico inicial página contratación_estado
# ===============================
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

response = requests.get(
    url_contratacion_estado,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)
print("URL final:", response.url)
print("Tamaño HTML:", len(response.text))

Status code: 200
URL final: https://contrataciondelestado.es/FileSystem/servlet/GetDocumentByIdServlet?DocumentIdParam=DYMMp2mFqXz4PkjvZh9I5vyWLItwtIt0n8yxEgEwqVaUFQkv5PXRMxoYGegc0NMYm0UHT%2FoXEFKTkr37fnFcEQrX2pafwfyc3CjQTJA0783VTD3T98KC0nSgQFM0q%2B5s&cifrado=QUC1GjXXSiLkydRHJBmbpw%3D%3D
Tamaño HTML: 88627


In [46]:
# ===============================
# 10.1 Diagnosticar tipo de contenido
# ===============================

print("Status code:", response.status_code)
print("URL final:", response.url)
print("Content-Type:", response.headers.get("Content-Type"))
print("Content-Disposition:", response.headers.get("Content-Disposition"))
print("Tamaño contenido:", len(response.content))

print("\nPrimeros 500 caracteres:")
print(response.text[:500])

Status code: 200
URL final: https://contrataciondelestado.es/FileSystem/servlet/GetDocumentByIdServlet?DocumentIdParam=DYMMp2mFqXz4PkjvZh9I5vyWLItwtIt0n8yxEgEwqVaUFQkv5PXRMxoYGegc0NMYm0UHT%2FoXEFKTkr37fnFcEQrX2pafwfyc3CjQTJA0783VTD3T98KC0nSgQFM0q%2B5s&cifrado=QUC1GjXXSiLkydRHJBmbpw%3D%3D
Content-Type: application/pdf
Content-Disposition: inline;filename=DOC_FORM2025-000964648.pdf
Tamaño contenido: 93319

Primeros 500 caracteres:
%PDF-1.4
%����
2 0 obj <</ColorSpace[/Indexed/DeviceRGB 255(���P��T����������������������������������z�������������������������i�����������������������j������P��������i�����i����������������|��������������M��S�����n������������������Y�������������폼�T��\\�����u��p�����Z�����{����������������{�����x�����������������}��~��j����������j�������������������p�˦��������z�����������������������������������������������������������������������������������������������������������������������t��������������


In [47]:
# ===============================
# 10.21 Esquema amplio para contratación_estado
# ===============================

campos_generales_esperados = [
    # Identificación
    "licitacion_id",
    "portal",
    "url_original",
    "url_final",
    "tipo_anuncio",
    "numero_expediente",
    "fecha_publicacion",
    "hora_publicacion",

    # Regulación
    "contrato_sujeto_regulacion_armonizada",
    "directiva_aplicacion",

    # Entidad adjudicadora
    "entidad_adjudicadora",
    "tipo_administracion",
    "actividad_principal",
    "tipo_entidad_adjudicadora",
    "perfil_contratante",
    "telefono_entidad",
    "fax_entidad",
    "email_entidad",
    "direccion_entidad",
    "codigo_subentidad_entidad",

    # Objeto general
    "objeto_contrato",
    "descripcion_general",
    "valor_estimado_contrato",
    "presupuesto_base_importe",
    "presupuesto_base_sin_impuestos",
    "cpv_codes",
    "cpv_descripciones",
    "plazo_ejecucion_inicio",
    "plazo_ejecucion_fin",
    "lugar_ejecucion",
    "codigo_subentidad_territorial",

    # Lotes
    "num_lotes",
    "num_lotes_resultado",
    "se_debe_ofertar",
    "max_lotes_presentacion",
    "max_lotes_adjudicacion",

    # Proceso
    "procedimiento",
    "tramitacion",
    "tramitacion_gasto",
    "sistema_contratacion",
    "presentacion_oferta",
    "plazo_obtencion_pliegos",
    "plazo_presentacion_oferta",
    "tipo_acto_apertura",
    "fecha_apertura_oferta",
    "hora_apertura_oferta",
    "detalle_licitacion_url",

    # Financiación
    "programas_financiacion",

    # Metadatos documento
    "id_documento",
    "uuid_documento",
    "sello_tiempo"
]

In [48]:
# ===============================
# 10.22 Funciones auxiliares robustas
# ===============================

import re
import pandas as pd


def buscar_regex(texto, patron, flags=re.IGNORECASE):
    match = re.search(patron, texto, flags)

    if match:
        return match.group(1).strip()

    return None


def limpiar_texto(valor):
    if valor is None:
        return None

    valor = re.sub(r"\s+", " ", str(valor)).strip()
    return valor


def convertir_importe_eur(valor):
    if pd.isna(valor):
        return None

    valor = str(valor).strip()
    valor = valor.replace("EUR.", "")
    valor = valor.replace("EUR", "")
    valor = valor.replace(".", "")
    valor = valor.replace(",", ".")
    valor = valor.strip()

    try:
        return float(valor)
    except ValueError:
        return None


def extraer_fecha_hora_publicacion(texto):
    patron = (
        r"Publicado en la Plataforma de Contratación del Sector Público "
        r"el\s+([0-9]{2}-[0-9]{2}-[0-9]{4})\s+a\s+las\s+([0-9]{2}:[0-9]{2})"
    )

    match = re.search(patron, texto, flags=re.IGNORECASE)

    if match:
        return match.group(1), match.group(2)

    return None, None


def extraer_bloque(texto, inicio, finales):
    patron = rf"{inicio}\s*(.*?)(?:{'|'.join(finales)})"

    return buscar_regex(
        texto,
        patron,
        flags=re.IGNORECASE | re.DOTALL
    )

In [49]:
# ===============================
# 10.23 Extraer ficha general amplia
# ===============================

texto = texto_pdf_completo

fecha_publicacion_txt, hora_publicacion = extraer_fecha_hora_publicacion(texto)

cpv_matches = re.findall(
    r"(\b\d{8}\b)\s*-\s*([^\n]+)",
    texto
)

cpv_codes = [codigo for codigo, _ in cpv_matches]
cpv_descripciones = [descripcion.strip() for _, descripcion in cpv_matches]

perfil_contratante = buscar_regex(
    texto,
    r"Perfil del Contratante\s*(https?://[^\s]+)"
)

detalle_licitacion_url = buscar_regex(
    texto,
    r"(https://contrataciondelestado\.es/wps/poc\?uri=deeplink:detalle_licitacion[^\s]+)"
)

fecha_apertura, hora_apertura = None, None

match_apertura = re.search(
    r"El día\s+([0-9]{2}/[0-9]{2}/[0-9]{4})\s+a las\s+([0-9]{2}:[0-9]{2})",
    texto,
    flags=re.IGNORECASE
)

if match_apertura:
    fecha_apertura = match_apertura.group(1)
    hora_apertura = match_apertura.group(2)


ficha_general = {
    "licitacion_id": licitacion_id_contratacion_estado,
    "portal": "contratacion_estado",
    "url_original": url_contratacion_estado,
    "url_final": response.url,

    # Identificación
    "tipo_anuncio": buscar_regex(
        texto,
        r"(Anuncio de formalización de contrato|Anuncio de licitación|Anuncio de adjudicación)"
    ),
    "numero_expediente": buscar_regex(
        texto,
        r"Número de Expediente\s+([^\n]+)"
    ),
    "fecha_publicacion_texto": fecha_publicacion_txt,
    "hora_publicacion": hora_publicacion,

    # Regulación
    "contrato_sujeto_regulacion_armonizada": buscar_regex(
        texto,
        r"Contrato Sujeto a regulación armonizada\s+([^\n]+)"
    ),
    "directiva_aplicacion": buscar_regex(
        texto,
        r"Directiva de aplicación\s*([^\n]+)"
    ),

    # Entidad adjudicadora
    "entidad_adjudicadora": buscar_regex(
        texto,
        r"Entidad Adjudicadora\s*([^\n]+)"
    ),
    "tipo_administracion": buscar_regex(
        texto,
        r"Tipo de Administración\s+([^\n]+)"
    ),
    "actividad_principal": buscar_regex(
        texto,
        r"Actividad Principal\s+([^\n]+)"
    ),
    "tipo_entidad_adjudicadora": buscar_regex(
        texto,
        r"Tipo de Entidad Adjudicadora\s+([^\n]+)"
    ),
    "perfil_contratante": perfil_contratante,
    "telefono_entidad": buscar_regex(
        texto,
        r"Teléfono\s+([0-9+ ]+)"
    ),
    "fax_entidad": buscar_regex(
        texto,
        r"Fax\s+([0-9+ ]+)"
    ),
    "email_entidad": buscar_regex(
        texto,
        r"Correo Electrónico\s+([^\s]+@[^\s]+)"
    ),

    # Objeto general
    "objeto_contrato": buscar_regex(
        texto,
        r"Objeto del Contrato:\s*([^\n]+)"
    ),
    "descripcion_general": buscar_regex(
        texto,
        r"Descripción\s+(.*?)(?:Valor estimado del contrato)",
        flags=re.IGNORECASE | re.DOTALL
    ),
    "valor_estimado_contrato": buscar_regex(
        texto,
        r"Valor estimado del contrato\s+([0-9\.\,]+)\s+EUR"
    ),
    "presupuesto_base_importe": buscar_regex(
        texto,
        r"Presupuesto base de licitación\s+Importe\s+([0-9\.\,]+)\s+EUR"
    ),
    "presupuesto_base_sin_impuestos": buscar_regex(
        texto,
        r"Importe \(sin impuestos\)\s+([0-9\.\,]+)\s+EUR"
    ),
    "cpv_codes": list(dict.fromkeys(cpv_codes)),
    "cpv_descripciones": list(dict.fromkeys(cpv_descripciones)),
    "plazo_ejecucion_inicio": buscar_regex(
        texto,
        r"Plazo de Ejecución\s+Del\s+([0-9]{2}/[0-9]{2}/[0-9]{4})"
    ),
    "plazo_ejecucion_fin": buscar_regex(
        texto,
        r"Plazo de Ejecución\s+Del\s+[0-9]{2}/[0-9]{2}/[0-9]{4}\s+al\s+([0-9]{2}/[0-9]{2}/[0-9]{4})"
    ),
    "lugar_ejecucion": buscar_regex(
        texto,
        r"Lugar de ejecución\s+Subentidad Nacional\s+([^\n]+)"
    ),
    "codigo_subentidad_territorial": buscar_regex(
        texto,
        r"Código de Subentidad Territorial\s+([A-Z0-9]+)"
    ),

    # Lotes
    "num_lotes": buscar_regex(
        texto,
        r"Nº de Lotes:\s*([0-9]+)"
    ),
    "num_lotes_resultado": buscar_regex(
        texto,
        r"Nº de Lotes cuyo resultado se indica en este anuncio:\s*([0-9]+)"
    ),
    "se_debe_ofertar": buscar_regex(
        texto,
        r"Se debe ofertar:\s*([^\n]+)"
    ),
    "max_lotes_presentacion": buscar_regex(
        texto,
        r"Número máximo de lotes a los que se puede presentar:?\s*([0-9]+)"
    ),
    "max_lotes_adjudicacion": buscar_regex(
        texto,
        r"Número máximo de lotes que se puede adjudicar a un licitador:?\s*([0-9]+)"
    ),

    # Proceso de licitación
    "procedimiento": buscar_regex(
        texto,
        r"Procedimiento\s+([^\n]+)"
    ),
    "tramitacion": buscar_regex(
        texto,
        r"Tramitación\s+([^\n]+)"
    ),
    "tramitacion_gasto": buscar_regex(
        texto,
        r"Tramitación del Gasto\s+([^\n]+)"
    ),
    "sistema_contratacion": buscar_regex(
        texto,
        r"Sistema de Contratación\s+([^\n]+)"
    ),
    "presentacion_oferta": buscar_regex(
        texto,
        r"Presentación de la oferta\s+([^\n]+)"
    ),
    "plazo_obtencion_pliegos": buscar_regex(
        texto,
        r"Plazo de Obtención de Pliegos\s+Hasta el\s+([^\n]+)"
    ),
    "plazo_presentacion_oferta": buscar_regex(
        texto,
        r"Plazo de Presentación de Oferta\s+Hasta el\s+([^\n]+)"
    ),
    "tipo_acto_apertura": buscar_regex(
        texto,
        r"Tipo de Acto\s*:\s*([^\n]+)"
    ),
    "fecha_apertura_oferta": fecha_apertura,
    "hora_apertura_oferta": hora_apertura,
    "detalle_licitacion_url": detalle_licitacion_url,

    # Financiación
    "programas_financiacion": buscar_regex(
        texto,
        r"Programas de Financiación\s+([^\n]+)"
    ),

    # Metadatos
    "id_documento": buscar_regex(
        texto,
        r"ID\s+([0-9]+)"
    ),
    "uuid_documento": buscar_regex(
        texto,
        r"UUID\s+([0-9\-]+)"
    ),
    "sello_tiempo": buscar_regex(
        texto,
        r"SELLO DE TIEMPO\s+(.+?)\s+N\.Serie",
        flags=re.IGNORECASE | re.DOTALL
    )
}

df_contratacion_general = pd.DataFrame([ficha_general])

# Limpieza texto
for columna in df_contratacion_general.columns:
    df_contratacion_general[columna] = df_contratacion_general[columna].apply(limpiar_texto)

# Conversión numérica de importes
for columna in [
    "valor_estimado_contrato",
    "presupuesto_base_importe",
    "presupuesto_base_sin_impuestos"
]:
    df_contratacion_general[columna + "_num"] = (
        df_contratacion_general[columna].apply(convertir_importe_eur)
    )

# Fechas
df_contratacion_general["fecha_publicacion"] = pd.to_datetime(
    df_contratacion_general["fecha_publicacion_texto"],
    format="%d-%m-%Y",
    errors="coerce"
)

df_contratacion_general.T

,0
licitacion_id,ddec9f7e2e9944dc
portal,contratacion_estado
url_original,https://contrataciondelestado.es/FileSystem/se...
url_final,https://contrataciondelestado.es/FileSystem/se...
tipo_anuncio,Anuncio de formalización de contrato
numero_expediente,40/VC-184/25
fecha_publicacion_texto,25-08-2025
hora_publicacion,10:20
contrato_sujeto_regulacion_armonizada,No
directiva_aplicacion,N/A


In [50]:
# ===============================
# 10.24 Detectar bloques de lote
# ===============================

patron_lote = r"Nº Lote:\s*([0-9]+)"

matches_lotes = list(re.finditer(patron_lote, texto, flags=re.IGNORECASE))

bloques_lotes = []

for i, match in enumerate(matches_lotes):
    inicio = match.start()

    if i + 1 < len(matches_lotes):
        fin = matches_lotes[i + 1].start()
    else:
        fin = texto.find("Proceso de Licitación")

        if fin == -1:
            fin = len(texto)

    texto_lote = texto[inicio:fin]

    bloques_lotes.append({
        "num_lote": match.group(1),
        "texto_lote": texto_lote
    })

print("Lotes detectados:", len(bloques_lotes))

Lotes detectados: 3


In [51]:
# ===============================
# 10.25 Extraer información por lote
# ===============================

registros_lotes = []

for bloque in bloques_lotes:
    texto_lote = bloque["texto_lote"]

    cpv_lote = re.findall(
        r"(\b\d{8}\b)\s*-\s*([^\n]+)",
        texto_lote
    )

    registro = {
        "licitacion_id": licitacion_id_contratacion_estado,
        "numero_expediente": ficha_general["numero_expediente"],
        "num_lote": bloque["num_lote"],

        "objeto_lote": buscar_regex(
            texto_lote,
            r"Objeto del Contrato:\s*([^\n]+)"
        ),
        "descripcion_lote": buscar_regex(
            texto_lote,
            r"Descripción\s+([^\n]+)"
        ),
        "valor_estimado_lote": buscar_regex(
            texto_lote,
            r"Valor estimado del contrato\s+([0-9\.\,]+)\s+EUR"
        ),
        "presupuesto_base_lote": buscar_regex(
            texto_lote,
            r"Presupuesto base de licitación\s+Importe\s+([0-9\.\,]+)\s+EUR"
        ),
        "presupuesto_base_lote_sin_impuestos": buscar_regex(
            texto_lote,
            r"Importe \(sin impuestos\)\s+([0-9\.\,]+)\s+EUR"
        ),

        "lugar_ejecucion_lote": buscar_regex(
            texto_lote,
            r"Lugar de ejecución\s+Subentidad Nacional\s+([^\n]+)"
        ),
        "codigo_subentidad_lote": buscar_regex(
            texto_lote,
            r"Código de Subentidad Territorial\s+([A-Z0-9]+)"
        ),
        "estado_lote": buscar_regex(
            texto_lote,
            r"(Formalizado|Adjudicado|Desierto|Anulado)"
        ),

        # Ofertas
        "precio_oferta_mas_baja": buscar_regex(
            texto_lote,
            r"Precio de la oferta más baja\s+([0-9\.\,]+)\s+EUR"
        ),
        "precio_oferta_mas_alta": buscar_regex(
            texto_lote,
            r"Precio de la oferta más alta\s+([0-9\.\,]+)\s+EUR"
        ),
        "num_ofertas_pymes": buscar_regex(
            texto_lote,
            r"Número de ofertas recibidas de PYMEs\s+([0-9]+)"
        ),

        # Adjudicatario
        "adjudicatario": buscar_regex(
            texto_lote,
            r"Adjudicatario\s+([^\n]+)"
        ),
        "nif_adjudicatario": buscar_regex(
            texto_lote,
            r"NIF\s+([A-Z0-9]+)"
        ),
        "adjudicatario_pyme": buscar_regex(
            texto_lote,
            r"El adjudicatario es una PYME\s*:\s*([^\n]+)"
        ),
        "direccion_adjudicatario": buscar_regex(
            texto_lote,
            r"Dirección Física\s+(.+?)\s+Contacto",
            flags=re.IGNORECASE | re.DOTALL
        ),
        "telefono_adjudicatario": buscar_regex(
            texto_lote,
            r"Contacto\s+Teléfono\s+([+0-9 ]+)"
        ),
        "email_adjudicatario": buscar_regex(
            texto_lote,
            r"Correo Electrónico\s+([^\s]+@[^\s]+)"
        ),

        # Adjudicación
        "importe_adjudicacion_sin_impuestos": buscar_regex(
            texto_lote,
            r"Importe total ofertado \(sin impuestos\)\s+([0-9\.\,]+)\s+EUR"
        ),
        "importe_adjudicacion_con_impuestos": buscar_regex(
            texto_lote,
            r"Importe total ofertado \(con impuestos\)\s+([0-9\.\,]+)\s+EUR"
        ),

        # Contrato
        "numero_contrato": buscar_regex(
            texto_lote,
            r"Número de contrato\s+([^\n]+)"
        ),
        "fecha_formalizacion": buscar_regex(
            texto_lote,
            r"Fecha de Formalización\s+([0-9]{2}/[0-9]{2}/[0-9]{4})"
        ),
        "documento_contrato": buscar_regex(
            texto_lote,
            r"Contrato\s+([^\n]+\.pdf)"
        ),
        "fecha_entrada_vigor": buscar_regex(
            texto_lote,
            r"Fecha de Entrada en Vigor del Contrato\s+([0-9]{2}/[0-9]{2}/[0-9]{4})"
        ),
        "motivacion": buscar_regex(
            texto_lote,
            r"Motivación\s+(.+?)\s+Fecha del Acuerdo de Adjudicación",
            flags=re.IGNORECASE | re.DOTALL
        ),
        "fecha_acuerdo_adjudicacion": buscar_regex(
            texto_lote,
            r"Fecha del Acuerdo de Adjudicación\s+([0-9]{2}/[0-9]{2}/[0-9]{4})"
        ),

        # CPV si aparece por lote
        "cpv_codes_lote": [codigo for codigo, _ in cpv_lote],
        "cpv_descripciones_lote": [descripcion for _, descripcion in cpv_lote],

        # Auditoría
        "texto_lote": texto_lote
    }

    registros_lotes.append(registro)

df_contratacion_lotes = pd.DataFrame(registros_lotes)

# Limpieza texto
for columna in df_contratacion_lotes.columns:
    if columna not in ["cpv_codes_lote", "cpv_descripciones_lote"]:
        df_contratacion_lotes[columna] = df_contratacion_lotes[columna].apply(limpiar_texto)

# Conversión numérica
columnas_importe_lote = [
    "valor_estimado_lote",
    "presupuesto_base_lote",
    "presupuesto_base_lote_sin_impuestos",
    "precio_oferta_mas_baja",
    "precio_oferta_mas_alta",
    "importe_adjudicacion_sin_impuestos",
    "importe_adjudicacion_con_impuestos"
]

for columna in columnas_importe_lote:
    df_contratacion_lotes[columna + "_num"] = (
        df_contratacion_lotes[columna].apply(convertir_importe_eur)
    )

# Fechas
for columna in [
    "fecha_formalizacion",
    "fecha_entrada_vigor",
    "fecha_acuerdo_adjudicacion"
]:
    df_contratacion_lotes[columna + "_dt"] = pd.to_datetime(
        df_contratacion_lotes[columna],
        format="%d/%m/%Y",
        errors="coerce"
    )

df_contratacion_lotes[
    [
        "num_lote",
        "objeto_lote",
        "descripcion_lote",
        "valor_estimado_lote_num",
        "adjudicatario",
        "nif_adjudicatario",
        "importe_adjudicacion_sin_impuestos_num",
        "importe_adjudicacion_con_impuestos_num",
        "fecha_formalizacion_dt",
        "fecha_acuerdo_adjudicacion_dt"
    ]
]

,num_lote,objeto_lote,descripcion_lote,valor_estimado_lote_num,adjudicatario,nif_adjudicatario,importe_adjudicacion_sin_impuestos_num,importe_adjudicacion_con_impuestos_num,fecha_formalizacion_dt,fecha_acuerdo_adjudicacion_dt
0,2,Reconocimiento médico otorrinolaringológico,Valor estimado del contrato 3.960 EUR.,3960.0,"TILOSALUD, S.A.",A40213217,3630.0,3630.0,NaT,2025-08-11
1,3,Reconocimiento médico ginecológico,Valor estimado del contrato 5.400 EUR.,5400.0,"TILOSALUD, S.A.",A40213217,4680.0,4680.0,NaT,2025-08-11
2,4,Reconocimiento médico urológico,Valor estimado del contrato 1.575 EUR.,1575.0,"TILOSALUD, S.A.",A40213217,1449.0,1449.0,NaT,2025-08-11


In [52]:
# ===============================
# 10.26 Extraer criterios de adjudicación
# ===============================

patron_criterio = re.compile(
    r"(?P<criterio>.+?)\s+"
    r"Subtipo Criterio\s*:\s*(?P<subtipo>[^\n]+)\s+"
    r"Ponderación\s*:\s*(?P<ponderacion>[0-9]+)"
    r"(?:\s+Expresión de evaluación\s*:\s*(?P<expresion>.+?))?"
    r"\s+Cantidad Mínima\s*:\s*(?P<cantidad_minima>[0-9]+)"
    r"\s+Cantidad Máxima\s*:\s*(?P<cantidad_maxima>[0-9]+)",
    flags=re.IGNORECASE | re.DOTALL
)

criterios = []

for match in patron_criterio.finditer(texto):
    criterio = limpiar_texto(match.group("criterio"))

    # Limpieza para evitar arrastrar encabezados demasiado largos
    criterio = re.split(
        r"Condiciones de adjudicación|Criterios de Adjudicación|Criterios evaluables mediante aplicación de fórmulas",
        criterio,
        flags=re.IGNORECASE
    )[-1].strip()

    criterios.append({
        "licitacion_id": licitacion_id_contratacion_estado,
        "numero_expediente": ficha_general["numero_expediente"],
        "criterio": criterio,
        "subtipo_criterio": limpiar_texto(match.group("subtipo")),
        "ponderacion": int(match.group("ponderacion")),
        "expresion_evaluacion": limpiar_texto(match.group("expresion")),
        "cantidad_minima": int(match.group("cantidad_minima")),
        "cantidad_maxima": int(match.group("cantidad_maxima"))
    })

df_contratacion_criterios = pd.DataFrame(criterios)

df_contratacion_criterios

""


In [53]:
# ===============================
# 10.27 Validaciones de extracción
# ===============================

validaciones = {
    "valor_estimado_general": df_contratacion_general.loc[0, "valor_estimado_contrato_num"],
    "presupuesto_base_general": df_contratacion_general.loc[0, "presupuesto_base_importe_num"],
    "num_lotes_reportado": df_contratacion_general.loc[0, "num_lotes"],
    "num_lotes_resultado_reportado": df_contratacion_general.loc[0, "num_lotes_resultado"],
    "num_lotes_extraidos": df_contratacion_lotes.shape[0],
    "suma_valor_estimado_lotes_extraidos": df_contratacion_lotes["valor_estimado_lote_num"].sum(),
    "suma_adjudicacion_sin_impuestos": df_contratacion_lotes["importe_adjudicacion_sin_impuestos_num"].sum(),
    "suma_adjudicacion_con_impuestos": df_contratacion_lotes["importe_adjudicacion_con_impuestos_num"].sum(),
    "num_criterios_extraidos": df_contratacion_criterios.shape[0]
}

validaciones

{'valor_estimado_general': np.float64(15975.0),
 'presupuesto_base_general': np.float64(15975.0),
 'num_lotes_reportado': '4',
 'num_lotes_resultado_reportado': '3',
 'num_lotes_extraidos': 3,
 'suma_valor_estimado_lotes_extraidos': np.float64(10935.0),
 'suma_adjudicacion_sin_impuestos': np.float64(9759.0),
 'suma_adjudicacion_con_impuestos': np.float64(9759.0),
 'num_criterios_extraidos': 0}